# ML DEPELYMENT AND LIFECYCLE MANAGEMNET
## Section 3

Assignment questions and answers


## Question 1

Install Flask in a new Python virtual environment and create a file called app.py that displays 'Welcome to the Prediction API' at the root URL (/).

**Answer:**

### Step 1: Set up the Virtual Environment and Install Flask

Run these commands in your terminal:

```bash
# Create a virtual environment named 'venv'
python -m venv venv

# Activate the virtual environment
# On Windows:
venv\Scripts\activate
# On macOS/Linux:
source venv/bin/activate

# Install Flask
pip install Flask
```

---

### Step 2: Create `app.py`

Create a file named `app.py` and add the following code:

```python
from flask import Flask

# Initialize the Flask application
app = Flask(__name__)

# Define the route for the root URL
@app.route('/')
def home():
    return 'Welcome to the Prediction API'

# Run the app
if __name__ == '__main__':
    app.run(debug=True)
```

### Explanation

* `app = Flask(__name__)` creates an instance of the Flask web server.
* `@app.route('/')` tells Flask to run the `home()` function whenever someone visits the root URL (`http://127.0.0.1:5000/`).
* `return 'Welcome to the Prediction API'` sends that text to the user's browser.
* You can run this app by typing `python app.py` in your terminal.

## Question 2

Build a simple REST API in Flask called /predict-price that accepts a POST request with JSON input containing 'base_price' and 'discount' fields, and returns the final price after applying the discount.

**Answer:**

Here is the Python code using Flask to create the API:

```python
from flask import Flask, jsonify, request

app = Flask(__name__)


@app.route("/predict-price", methods=["POST"])
def predict_price():
    # Read the JSON data sent in the POST request
    data = request.get_json()

    base_price = data.get("base_price", 0)
    discount = data.get("discount", 0)  # Discount as a percentage (e.g., 10 for 10%)

    # Calculate final price after discount
    final_price = base_price - (base_price * (discount / 100))

    # Return the result as a JSON response
    return jsonify({"final_price": round(final_price, 2)})


if __name__ == "__main__":
    app.run(debug=True)
```

### Explanation:
- **`request.get_json()`**: Extracts the incoming JSON payload into a Python dictionary.
- **Calculation**: Subtracts the discount percentage from the base price (`base_price - (base_price * (discount / 100))`).
- **`jsonify()`**: Converts the result dictionary back into a JSON response.

### Example Request Body (JSON):
```json
{
  "base_price": 100.0,
  "discount": 15
}
```

### Example Response (JSON):
```json
{
  "final_price": 85.0
}
```

## Question 3

Save a trained scikit-learn LinearRegression model (for example, predicting delivery time based on distance and order size) to a file using joblib, then load this model in your Flask app and use it to make predictions when the /predict-delivery endpoint receives a POST request with JSON input.

**Answer:**

Here is how to train and save the model, then serve it using Flask.

### Step 1: Train and Save the Model

Run this script first to train the model on sample data and save it to a file.

```python
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression

# Invented sample data
sample_data = pd.DataFrame(
    {
        "distance": [2, 5, 8, 1, 10],  # in miles
        "order_size": [1, 3, 4, 1, 6],  # number of items
        "delivery_time": [15, 30, 45, 12, 60],  # in minutes
    }
)

# Separate features and target
X = sample_data[["distance", "order_size"]]
y = sample_data["delivery_time"]

# Train the model
model = LinearRegression()
model.fit(X, y)

# Save the trained model to a file
joblib.dump(model, "delivery_model.pkl")
print("Model saved as delivery_model.pkl")
```

### Step 2: Create the Flask App

This script loads the saved model and predicts delivery time when a POST request is sent to `/predict-delivery`.

```python
import joblib
from flask import Flask, jsonify, request

app = Flask(__name__)

# Load the saved model
model = joblib.load("delivery_model.pkl")


@app.route("/predict-delivery", methods=["POST"])
def predict():
    # Read JSON data from the request
    data = request.get_json()

    distance = data["distance"]
    order_size = data["order_size"]

    # Predict using the model (expects a 2D array)
    prediction = model.predict([[distance, order_size]])

    # Return the result as JSON
    return jsonify({"predicted_delivery_time": round(prediction[0], 2)})


if __name__ == "__main__":
    app.run(debug=True)
```

### Explanation

1. **Training (`joblib.dump`)**: We fit a `LinearRegression` model using sample data and save it into a file named `delivery_model.pkl`.
2. **Loading (`joblib.load`)**: The Flask app loads the file into memory once when starting up.
3. **Prediction**: The `/predict-delivery` route takes the input values (`distance` and `order_size`) from an incoming JSON body, passes them to `model.predict()`, and returns the predicted time as a JSON response. 

**Example JSON Request Body:**
```json
{
  "distance": 4,
  "order_size": 2
}
```

## Question 4

Update your Flask API so that it returns a JSON response with both the predicted value and a custom message, similar to how Swiggy or Zomato shows estimated delivery time and a friendly note.

**Answer:**

Here is the updated Flask code that returns both the predicted time and a friendly message using `jsonify`.

### Python Code

```python
from flask import Flask, jsonify

app = Flask(__name__)


@app.route("/predict", methods=["GET"])
def predict_delivery():
    # Sample data: simulated prediction value
    predicted_time = 30

    # Custom friendly message
    custom_message = (
        "Hang tight! Our delivery partner is rushing to bring your warm meal."
    )

    # Return both values as a JSON response
    return jsonify(
        {
            "status": "success",
            "estimated_delivery_time": f"{predicted_time} mins",
            "message": custom_message,
        }
    )


if __name__ == "__main__":
    app.run(debug=True)
```

### Explanation
We use Flask's `jsonify` function to return a Python dictionary as a JSON object. Inside the dictionary, we pass two main keys: `estimated_delivery_time` for the predicted value and `message` for the friendly notification.

## Question 5

Run your Flask prediction API locally, then use Postman or curl to send a sample JSON request to your /predict-delivery endpoint and verify the response.

Hint: If you get a CORS or content-type error, check your request headers and Flask route methods.</em

**Answer:**

Here is how you can set up, run, and test your Flask prediction API.

### 1. Python Code (`app.py`)
This script creates a local API with a `/predict-delivery` endpoint that accepts POST requests.

```python
from flask import Flask, jsonify, request

app = Flask(__name__)


@app.route("/predict-delivery", methods=["POST"])
def predict():
    # Read the JSON data sent in the request
    data = request.get_json()

    # Sample logic: calculate delivery time based on distance (Sample data)
    distance = data.get("distance", 0)
    predicted_time = distance * 5 + 10  # 5 mins per km + 10 mins prep

    return jsonify(
        {"status": "success", "predicted_delivery_minutes": predicted_time}
    )


if __name__ == "__main__":
    app.run(debug=True, port=5000)
```

### 2. Run the API
Run this in your terminal to start the server:
```bash
python app.py
```

### 3. Send a Request using `curl`
Open a new terminal window and run this command:

```bash
curl -X POST http://127.0.0.1:5000/predict-delivery \
-H "Content-Type: application/json" \
-d "{\"distance\": 5}"
```

*(Note: In Postman, set the method to **POST**, enter URL `http://127.0.0.1:5000/predict-delivery`, go to the **Body** tab, select **raw** -> **JSON**, and paste `{"distance": 5}`).*

### 4. Verified Response
If it works, the server will return this JSON response:
```json
{
  "predicted_delivery_minutes": 35,
  "status": "success"
}
```